# Demo 4 --- Testing by design, not by anecdote

The first three demos showed hand-picked cases. The natural objection is: how do we know this holds beyond the examples on the slide? The answer is to name what breaks the system, cover those combinations by design, and run the whole suite through the governed agent. This closes on real suite output, not a scoreboard.

## Name what breaks the system

The failure-mode catalog is the list of adversarial and degenerate behaviors a test must exercise --- a prompt injection, a malformed call, a hallucinated citation, a wedged loop. A suite that only sends clean complaints never touches the gates, so these are what a designed suite injects on purpose.

In [ ]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')
from agentlab.evaluation import ALL_INJECTORS, DEFAULT_FACTORS, balanced_design, coverage_report

for mode, ctor in ALL_INJECTORS.items():
    print(f'{mode.value:22s} {ctor().description}')

## Cover the hard combinations by design

Rather than pick cases, enumerate the factors that make a complaint hard and cover their levels with a balanced design, so the suite is not skewed toward easy cases. Each factor level appears roughly equally across the generated cases.

In [ ]:
design = balanced_design(DEFAULT_FACTORS, num_cases=12, seed=1)
for i, case in enumerate(design):
    print(f'{i:2d}: {case}')
print('\ncoverage:', coverage_report(design, DEFAULT_FACTORS))

## Run the suite through the governed agent

The evaluation cases carry an expected outcome: the adversarial and escalation-required cases are expected to escalate, the routine ones to be answered. Running the whole suite through the harness and aggregating gives a property of the agent, not a verdict on one case.

In [ ]:
import json
from agentlab.capstone import build_complaint_harness
from agentlab.core import Budget, BudgetTracker, TaskSpec

root = next((c for c in (Path('.'), Path('..'), Path('../..'), Path('../../code'), Path('../code'))
             if (c / 'data' / 'eval_cases' / 'cases.json').exists()), Path('.'))
cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())
harness, registry = build_complaint_harness(policies_dir=root / 'data' / 'policies')

def did_escalate(traj):
    if traj.final_state.status in ('escalated', 'failed'):
        return True
    out = traj.final_state.final_output or {}
    return isinstance(out, dict) and out.get('recommended_action') == 'escalate'

rows, correct = [], 0
for case in cases:
    task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
    traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
    esc = did_escalate(traj)
    ok = (esc == case.get('expected_escalation'))
    correct += ok
    rows.append((case['id'], case.get('expected_escalation'), esc, ok))

print(f"{'case':10s} {'expect_esc':11s} {'got_esc':8s} ok")
for cid, exp, got, ok in rows:
    print(f'{cid:10s} {str(exp):11s} {str(got):8s} {"OK" if ok else "XX"}')
print(f'\nescalation accuracy: {correct}/{len(cases)}')
print('audit chain valid  :', harness.audit.verify())

Each escalation traces to why it fired --- a PII refusal, a UDAAP flag, an unsafe draft --- so the aggregate is not a bare score but a statement about which designed stressors the agent handles and how. That is the evidence a hand-picked example cannot give: the suite covers the combinations no one wrote by hand, and the result is reproducible.